In [ ]:
import pandas as pd
lang="eu"
pos=f"./evaluation/benchmarks/recon/recon_io_{lang}.jsonl"
orig=f"./evaluation/benchmarks/recon/recon_io_{lang}_mt.jsonl"

In [ ]:
d=pd.read_json("./evaluation/benchmarks/recon/recon_en.jsonl", lines=True)
d_pos=pd.read_json(pos,lines=True)
d_orig=pd.read_json(orig,lines=True)

In [ ]:
def extract(prompt):
    aux=prompt.split("###The instruction to evaluate:")[1].split("###Response to evaluate:")
    instruction=aux[0].strip()
    response=aux[1].split("###Reference Answer (Score 5):")[0].strip()
    return instruction,response

In [ ]:
d[["instruction","response"]]=d.apply(lambda x: extract(x["inputs"]),axis=1, result_type="expand")
d_pos[["instruction","response"]]=d_pos.apply(lambda x: extract(x["inputs"]),axis=1, result_type="expand")
d_orig[["instruction","response"]]=d_orig.apply(lambda x: extract(x["inputs"]),axis=1, result_type="expand")

In [ ]:
import evaluate as evaluate
from comet import download_model, load_from_checkpoint

bleu = evaluate.load("bleu")
chrf = evaluate.load("chrf")

comet_model = load_from_checkpoint(download_model("Unbabel/wmt22-comet-da"))


def compare_columns(col1, col2, origin, name="comparison"):
    preds = col1.tolist()
    refs = col2.tolist()
    
    # BLEU
    bleu_score = bleu.compute(predictions=preds, references=refs)["bleu"]

    # chrF
    chrf_score = chrf.compute(predictions=preds, references=refs)["score"]

    # COMET
    comet_data = [
        {"src": o, "mt": p, "ref": r}
        for o, p, r in zip(origin,preds, refs)
    ]

    comet_scores = comet_model.predict(comet_data, batch_size=8, gpus=1)
    comet_avg = sum(comet_scores["scores"]) / len(comet_scores["scores"])

    print(f"\n{name}:")
    print(f"BLEU: {bleu_score:.4f}")
    print(f"chrF: {chrf_score:.4f}")
    print(f"COMET: {comet_avg:.4f}")

# ---- Compare across datasets ----

compare_columns(pd.concat([d_pos["instruction"],d_pos["response"]]),pd.concat([d_orig["instruction"],d_orig["response"]]), pd.concat([d["instruction"],d["response"]]), "Tr vs Tr")